In [18]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
movies=pd.read_csv("Netflix_Data_clean.csv",sep=';')
movies.head()

,show_id,type,title,director,cast,country,release_year,rating,listed_in,description,date_added_clean,duration_minutes,duration_seasons
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,2020,PG-13,Documentaries,"As her father nears the end of his life, filmm...",2021-09-25,90,0
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021,TV-MA,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t...",2021-09-24,0,2
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,2021,TV-MA,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...,2021-09-24,0,1
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,2021,TV-MA,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo...",2021-09-24,0,1
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021,TV-MA,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...,2021-09-24,0,2


In [19]:
# Replace NAN values with empty strings
movies['type'] = movies['type'].fillna('')
movies['title'] = movies['title'].fillna('')
movies['director'] = movies['director'].fillna('')
movies['cast'] = movies['cast'].fillna('')
movies['country'] = movies['country'].fillna('')
release_year = movies['release_year'].fillna(0)
rating = movies['rating'].fillna('')
movies['listed_in'] = movies['listed_in'].fillna('')
movies['description'] = movies['description'].fillna('')
movies['date_added_clean'] = movies['date_added_clean'].fillna('')
movies['duration_minutes'] = movies['duration_minutes'].fillna(0)
movies['duration_seasons'] = movies['duration_seasons'].fillna(0)



In [26]:
# Tokenize the text data using TfidfVectorizer
#Tokenization is breaking down the text data into smaller units, such as words or phrases, to analyze and process it more effectively. In this case, we are using TfidfVectorizer to convert the text data into numerical representations that can be used for similarity calculations.
tfidf_vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf_vectorizer.fit_transform(movies['description'].fillna('') + ' ' + 
movies['director'].fillna('') + ' ' + movies['cast'].fillna('') + ' ' + 
movies['country'].fillna('') + ' ' + movies['listed_in'].fillna(''))
tfidf_matrix.shape

(100, 2458)

In [27]:
# Calculate the cosine similarity between the movies based on their descriptions
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
cosine_sim.shape    

(100, 100)

In [22]:
#Create a function to get movie recommendations based on the cosine similarity scores
indices = pd.Series(movies.index, index=movies['title']).drop_duplicates()
def get_recommendations(title, cosine_sim=cosine_sim,num_recommended=10):
    idx=indices[title]
    sim_scores=list(enumerate(cosine_sim[idx]))
    sim_scores=sorted(sim_scores,key=lambda x:x[1],reverse=True)
    top_10=sim_scores[1:num_recommended+1]
    movie_indices=[i[0] for i in top_10]
    return movies['title'].iloc[movie_indices]

In [25]:
get_recommendations('Squid Game')

34                   Tayo and Little Wizards
69            Stories by Rabindranath Tagore
86                             Titipo Titipo
70                 Too Hot To Handle: Latino
74                    You vs. Wild: Out Cold
25                      Love on the Spectrum
2                                  Ganglands
96                                Kid Cosmic
10       Vendetta: Truth, Lies and The Mafia
40    He-Man and the Masters of the Universe
Name: title, dtype: object